In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 3, Finished, Available, Finished, False)

## Silver tables

In [13]:
# Read Silver tables
sales = spark.table("silver_sales_daily").alias("s")      # date, store_nbr, family, onpromotion, sales
stores = spark.table("silver_store_dim").alias("st")        # store_nbr, city, state, type, cluster
items = spark.table("silver_item_dim").alias("i")          # item_id, family
calendar = spark.table("silver_calendar_dim").alias("c")   # date, year, month, day_of_week, is_holiday, etc.

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 15, Finished, Available, Finished, False)

## Left join item_dim + sales

In [15]:
# Join sales with item_dim to get item_id
sales_with_item = sales.join(items, on="family", how="left")

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 17, Finished, Available, Finished, False)

In [16]:
display(sales_with_item)

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a12bbbd-1282-41ff-8de2-2434d754d26c)

In [17]:
# Join with store and calendar dimensions
base = (
    sales_with_item.alias("s")
    .join(stores.alias("st"), on="store_nbr", how="left")
    .join(calendar.alias("c"), on="date", how="left")
)

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 19, Finished, Available, Finished, False)

In [18]:
display(base)
# 'base' has one row per (date, store_nbr, family/item_id) with calendar + store info

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab218080-a7c7-4977-a393-64b4dd7fdb15)

## window and add lag / moving average features

In [19]:
# Define window by store + item ordered by date
w = Window.partitionBy("store_nbr", "item_id").orderBy("date")

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 21, Finished, Available, Finished, False)

In [11]:
# Apply row_number to see the ordering within each partition
window_check = base.withColumn("row_num", F.row_number().over(w))

# Display the result, focusing on one store/item to see the sequence
display(window_check.select("store_nbr", "item_id", "date", "sales", "row_num")
                    .orderBy("store_nbr", "item_id", "date"))

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eca91e5a-367b-4c75-a37c-88c1d5c66693)

In [20]:
# lag features: previous 7 / 14 / 28 days sales (by row index)

# Rolling moving averages (last 7 / 14 / 28 rows including today)
w_7 = w.rowsBetween(-6, 0)     # 7 days window (6 previous + current)[web:76]
w_14 = w.rowsBetween(-13, 0)   # 14 days window
w_28 = w.rowsBetween(-27, 0)   # 28 days window

feat = (
    base
    .withColumn("lag_1", F.lag("sales", 1).over(w))
    .withColumn("lag_7", F.lag("sales", 7).over(w))
    .withColumn("lag_14", F.lag("sales", 14).over(w))
    .withColumn("lag_28", F.lag("sales", 28).over(w))
    .withColumn("ma_7", F.avg("sales").over(w_7))
    .withColumn("ma_14", F.avg("sales").over(w_14))
    .withColumn("ma_28", F.avg("sales").over(w_28))
)


StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 22, Finished, Available, Finished, False)

In [21]:
display(feat)

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 63711ab3-451b-40bb-8ba5-fab6dd812f86)

## featured column

In [22]:
# features

fact_store_item_features = feat.select(
    F.col("date"),
    F.col("store_nbr"),
    F.col("item_id"),
    F.col("family"),
    F.col("sales"),
    F.col("onpromotion"),
    F.col("c.is_holiday").alias("is_holiday"),
    F.col("c.type").alias("holiday_type"),
    F.col("year"),
    F.col("month"),
    F.col("day_of_week"),
    F.col("city"),
    F.col("state"),
    F.col("st.type").alias("store_type"),
    F.col("cluster"),
    F.col("lag_1"),
    F.col("lag_7"),
    F.col("lag_14"),
    F.col("lag_28"),
    F.col("ma_7"),
    F.col("ma_14"),
    F.col("ma_28")
)

# drop very early rows where moving averages are null
fact_store_item_features = fact_store_item_features.filter(F.col("ma_28").isNotNull())

# Save as Delta table in the Lakehouse
fact_store_item_features.write.mode("overwrite").format("delta").saveAsTable("fact_store_item_features")

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 24, Finished, Available, Finished, False)

In [23]:
display(fact_store_item_features)

StatementMeta(, 24a0ef46-51e1-4dfc-b3e6-69782f44b785, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 014a1686-abd2-471b-a6e6-71c906ac86ff)